# USA Housing Price Prediction — Simple Linear Regression

This notebook loads the USA Housing dataset, performs basic data inspection and EDA, removes price outliers using the IQR method, and builds a **Simple Linear Regression (SLR)** model using **Avg. Area Income** to predict **Price**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid")

## 1. Load the dataset

Place `USA_Housing.csv` in the same folder as this notebook before running the following cell.

In [ ]:
file_path = "USA_Housing.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"'{file_path}' was not found. Place USA_Housing.csv in the notebook's working directory."
    )

df = pd.read_csv(file_path)
            

In [ ]:
print(f"Dataset shape: {df.shape}")
display(df.head(20))

In [ ]:
display(df.tail())

In [ ]:
print("Missing values:")
display(df.isnull().sum())

print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Address is not useful for the numerical regression model.
if "Address" in df.columns:
    df = df.drop(columns=["Address"])

display(df.head())

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 8))

numeric_df = df.select_dtypes(include="number")
sns.heatmap(numeric_df.corr(), annot=True, cmap="inferno", fmt=".2f")

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Price", bins=30, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Avg. Area Income", y="Price")
plt.title("Average Area Income vs House Price")
plt.xlabel("Average Area Income")
plt.ylabel("House Price")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(y=df["Price"])
plt.title("House Price Boxplot")
plt.ylabel("Price")
plt.tight_layout()
plt.show()

## 3. Detect and remove price outliers using IQR

In [ ]:
Q1 = df["Price"].quantile(0.25)
Q3 = df["Price"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["Price"] < lower_bound) | (df["Price"] > upper_bound)]
            
print(f"Number of price outliers: {len(outliers)}")
display(outliers.head())

In [ ]:
before = len(df)

df = df[(df["Price"] >= lower_bound) & (df["Price"] <= upper_bound)].copy()

after = len(df)
print(f"Rows before outlier removal: {before}")
print(f"Rows after outlier removal:  {after}")
print(f"Rows removed:               {before - after}")

In [ ]:
display(df.tail())

## 4. Simple Linear Regression (SLR)

For **Simple Linear Regression**, we must use exactly **one independent variable**. Here, the predictor is `Avg. Area Income` and the target is `Price`.

The original notebook used two column names inside one string, which caused a column-selection error. Using two predictors would be **Multiple Linear Regression**, not SLR.

In [ ]:
X = df[["Avg. Area Income"]]
y = df["Price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

In [ ]:
slr_model = LinearRegression()
slr_model.fit(X_train, y_train)

In [ ]:
y_pred = slr_model.predict(X_test)
y_pred[:5]

## 5. Model evaluation

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:,.2f}")
print(f"Mean Squared Error (MSE):  {mse:,.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
print(f"Intercept: {slr_model.intercept_:,.2f}")
print(f"Coefficient for Avg. Area Income: {slr_model.coef_[0]:,.2f}")

print(
    f"\nRegression equation:\n"
    f"Price = {slr_model.intercept_:,.2f} + "
    f"({slr_model.coef_[0]:,.2f} × Avg. Area Income)"
)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_test["Avg. Area Income"],
    y_test,
    label="Actual Data",
    alpha=0.7
)

# Sort x values so the regression line is drawn correctly.
sorted_indices = np.argsort(X_test["Avg. Area Income"].values)
x_sorted = X_test.iloc[sorted_indices]
y_sorted_pred = slr_model.predict(x_sorted)

plt.plot(
    x_sorted["Avg. Area Income"],
    y_sorted_pred,
    linewidth=2,
    label="Regression Line"
)

plt.xlabel("Average Area Income")
plt.ylabel("House Price")
plt.title("Simple Linear Regression: Income vs House Price")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Predict the price of a new house

Enter an average area income value. The model will predict the corresponding house price.

In [ ]:
income = float(input("Enter Average Area Income: "))

new_data = pd.DataFrame({"Avg. Area Income": [income]})
prediction = slr_model.predict(new_data)[0]

print(f"Predicted House Price: {prediction:,.2f}")